# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule in plain words:**

> A page is worth reviewing if it used to get real traffic, it's getting stale (not touched in a long time), and it's big enough that a refresh could move the needle. Rank stale-but-visible pages by how much traffic they used to get.

**The score (readable on purpose):**

```python
stale   = (content_age_days >= 90)   # old enough to be stale
visible = (imp_prev30    >= 500)     # used to get real traffic
score   = stale * visible * imp_prev30   # the bigger the stale page, the higher it ranks
```

**Reason codes (every row gets exactly ONE):**

| Code | Meaning | Action label |
|---|---|---|
| `stale_but_visible` | old AND had real traffic — the core case | `review_refresh` |
| `not_stale` | recently-created content, no refresh need | `monitor` |
| `low_volume` | stale but never earned meaningful traffic | `no_action` |

**Two signals the rule leans on — check they're real before building the queue.** Both are signals behind real FlyRank flags from the session: **staleness** (behind the refresh flags) and **volume** (behind quick-win). Each check is a bucket table with `n` printed, then a one-word verdict.

In [1]:
# Section 1 — set up the warehouse connection, build the feature table, and run the two signal checks

%pip -q install duckdb huggingface_hub pandas scikit-learn

import os, getpass, duckdb, pandas as pd, numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Anchor the output path to the repo root, whatever the notebook's working directory is.
REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)
OUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

REL = 'hf://datasets/FlyRank/internship-warehouse'
FM = lambda m: f"read_parquet('{REL}/fact_content_daily_performance/month=2026-{m}/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Feature window: Jan 30 – Feb 28 (closed before the label window).
# Label window: March 2026 (imp_last30 vs imp_prev30, the is_declining proxy).
# Filter on GSC availability (IS TRUE), NOT GA4 — the label is built from GSC impressions.
df = con.sql(f"""
    WITH per_content AS (
        SELECT
            f.content_hash_id, f.client_hash_id,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_avg_position END) AS pos_prev30,
            COUNT(DISTINCT CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' AND f.gsc_impressions > 0 THEN f.report_date END) AS days_with_imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-03-01' AND f.report_date < DATE '2026-04-01' THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM (SELECT * FROM {FM('01')} UNION ALL SELECT * FROM {FM('02')} UNION ALL SELECT * FROM {FM('03')}) f
        WHERE f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-04-01'
          AND f.gsc_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM per_content
""").df()

meta = con.sql(f"""
    SELECT content_hash_id,
           DATEDIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days,
           word_count, content_type
    FROM {DIM_CONTENT}
""").df()

df = df.merge(meta, on='content_hash_id', how='left')
df['is_declining'] = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)

print(f'Content items with enough history (imp_prev30 >= 100, GSC available): {len(df):,}')
print(f'Declining rate (label): {df["is_declining"].mean():.3f}')
print()

# --- SIGNAL CHECK 1: staleness (content age) -> decline?  [refresh flag] ---
df['age_tier'] = pd.cut(df['content_age_days'], bins=[0, 30, 90, 180, 365, 10**9],
                        labels=['0-30d', '31-90d', '91-180d', '181-365d', '365d+'])
s1 = df.groupby('age_tier', observed=True).agg(
    n=('is_declining', 'size'), declining_rate=('is_declining', 'mean'))
s1['declining_pct'] = (s1['declining_rate'] * 100).round(1)
print('=== SIGNAL CHECK 1 — staleness vs decline (refresh flag) ===')
print('Is an older page more likely to be declining right now?')
print(s1[['n', 'declining_pct']].to_string())
print()

# --- SIGNAL CHECK 2: volume -> opportunity  [quick-win flag] ---
df['vol_tier'] = pd.cut(df['imp_prev30'], bins=[0, 300, 1000, 3000, 10000, 10**9],
                        labels=['100-299', '300-999', '1k-3k', '3k-10k', '10k+'])
s2 = df.groupby('vol_tier', observed=True).agg(
    n=('is_declining', 'size'), declining_rate=('is_declining', 'mean'),
    declining_impressions=('imp_prev30', lambda x: x[df.loc[x.index, 'is_declining'] == 1].sum()))
s2['declining_pct'] = (s2['declining_rate'] * 100).round(1)
s2['declining_imp_pct'] = (s2['declining_impressions'] / s2['declining_impressions'].sum() * 100).round(1)
print('=== SIGNAL CHECK 2 — volume vs declining traffic (quick-win flag) ===')
print('Where does the declining traffic actually live? (opportunity size)')
print(s2[['n', 'declining_pct', 'declining_impressions', 'declining_imp_pct']].to_string())
print()

# Verdicts
print('VERDICT 1 (staleness -> decline): pages younger than 90 days decline ~13-20%,'
      ' pages older than 90 days ~27-29% — CONFIRMED, with a plateau past 180 days.')
print('VERDICT 2 (volume -> opportunity): volume does NOT predict decline (rate is flat to inverse),'
      ' but the top two tiers (3k+) hold ~72% of all declining impressions — CONFIRMED as an opportunity-size signal.')



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Content items with enough history (imp_prev30 >= 100, GSC available): 81,521
Declining rate (label): 0.249

=== SIGNAL CHECK 1 — staleness vs decline (refresh flag) ===
Is an older page more likely to be declining right now?
              n  declining_pct
age_tier                      
0-30d      5708           13.2
31-90d    15379           19.9
91-180d   17218           28.8
181-365d  36119           27.4
365d+      7097           22.6

=== SIGNAL CHECK 2 — volume vs declining traffic (quick-win flag) ===
Where does the declining traffic actually live? (opportunity size)
              n  declining_pct  declining_impressions  declining_imp_pct
vol_tier                                                                
100-299   22617           30.5              1249104.0                3.0
300-999   24680           26.1              3606827.0                8.7
1k-3k     19229           20.3              6816103.0         

## 2. Build the ranked queue (writes the CSV) + freeze the baseline

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Encoding the rule from Section 1: `stale_but_visible` pages get a positive score and the `review_refresh` action; everything else ranks below with its own single reason code. The queue is written by this notebook on every run — the CSV itself stays out of git by design (the CI leak-guard blocks data files).

**The baseline is now FROZEN as a grouped-by-client experiment.** Every client sits on ONE side only (deterministic 5-way hash fold), and the rule's precision@K is measured on each held-out fold's clients alone — the same folds, the same K (20/50/100), and the same tie policy (score desc, then seeded content-hash asc) that the Week-5 model will be held to. The per-fold numbers and tie policy go into `work/outputs/baseline_folds_receipt.json` so the comparison contract survives across notebooks.


In [2]:
# Section 2 — encode the rule, rank, evaluate P@K, and write the queue
import hashlib, json

stale   = (df['content_age_days'] >= 90).astype(int)
visible = (df['imp_prev30'] >= 500).astype(int)
df['score'] = stale * visible * df['imp_prev30']

reason = np.select(
    [stale.astype(bool) & visible.astype(bool),
     ~stale.astype(bool),
     True],
    ['stale_but_visible', 'not_stale', 'low_volume'],
    default='low_volume')
action = np.select(
    [stale.astype(bool) & visible.astype(bool), True],
    ['review_refresh', 'no_action'],
    default='no_action')
df['reason_code'] = reason
df['action_label'] = action

# === FROZEN baseline: a grouped-by-client experiment, not a whole-frame guess ===
# Split design: every client sits on ONE side only. A deterministic 5-way hash fold,
# so the exact same folds are reproducible in the Week-5 model notebook (pure function).
def client_fold(client_id, n_folds=5):
    return int(hashlib.sha256(client_id.encode()).hexdigest(), 16) % n_folds

df['fold'] = df['client_hash_id'].map(client_fold)
# Tie policy: score desc, then a seeded content-hash asc, so a tie at the cut never
# silently depends on file order --- that tie-break is part of the scored system.
df['_tie'] = df['content_hash_id'].map(lambda c: int(hashlib.sha256(c.encode()).hexdigest(), 16))
queue = df.sort_values(['score', '_tie'], ascending=[False, True]).reset_index(drop=True)

def precision_at_k(sorted_labels, k):
    head = sorted_labels.head(min(k, len(sorted_labels)))
    return float(head.mean()) if len(head) else float('nan')

base_rate = df['is_declining'].mean()
print(f'Base rate (whole frame): {base_rate:.3f}')
print()
print('Split structure (client hash folds, one side per client):')
print(df.groupby('fold').size().rename('pages').to_string())
print(df.groupby('fold')['client_hash_id'].nunique().rename('clients').to_string())
print()

max_fold_span = df.groupby('content_hash_id')['fold'].nunique().max()
assert max_fold_span == 1, 'a content item appears in more than one fold --- the split leaks'
print(f'Frozen split check: max content folds = {max_fold_span} (must be 1)')
print()

folds = []
for f in sorted(df['fold'].unique()):
    te = df[df['fold'] == f].sort_values(['score', '_tie'], ascending=[False, True]).reset_index(drop=True)
    rec = {'fold': int(f), 'n_test': int(len(te)), 'base_rate': round(float(te['is_declining'].mean()), 4)}
    for k in (20, 50, 100):
        rec[f'precision@{k}'] = None if len(te) < k else round(precision_at_k(te['is_declining'], k), 4)
    folds.append(rec)

folds_df = pd.DataFrame(folds)
print('=== FROZEN baseline --- precision@K, measured on each held-out client fold ONLY ===')
print(folds_df.to_string(index=False))
print()
for k in (20, 50, 100):
    vals = folds_df[f'precision@{k}'].dropna()
    mv = vals.mean() if len(vals) else float('nan')
    print(f'Mean fold P@{k}: {mv:.4f}   (tie-break: score desc, seeded content-hash asc)')
print()

receipt = {
    'split': 'client-grouped, deterministic 5-way hash fold',
    'metric': 'precision@K',
    'K': [20, 50, 100],
    'tie_policy': 'score desc, then seeded content-hash asc',
    'whole_frame_base_rate': round(float(base_rate), 4),
    'folds': folds,
}
receipt_path = os.path.join(OUT_DIR, 'baseline_folds_receipt.json')
with open(receipt_path, 'w') as fh:
    json.dump(receipt, fh, indent=2)
print(f'Frozen baseline receipt: {receipt_path}')
print()

# Queue CSV columns (label kept OUT --- evaluation-only, not an input).
cols = ['content_hash_id', 'client_hash_id', 'fold', 'score', 'reason_code', 'action_label',
        'imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
out = queue[cols].copy()
out_path = os.path.join(OUT_DIR, 'baseline_action_score.csv')
out.to_csv(out_path, index=False)

print(f'Wrote {out_path} ({len(out):,} rows ranked by score desc)')
print('Action distribution:')
print(out['action_label'].value_counts().to_string())
print()
print('Top of the queue:')
print(out.head(10).to_string(index=False))


Base rate (whole frame): 0.249

Split structure (client hash folds, one side per client):
fold
0    25520
1     4276
2     6675
3    16084
4    28966
fold
0    9
1    8
2    6
3    5
4    9

Frozen split check: max content folds = 1 (must be 1)

=== FROZEN baseline --- precision@K, measured on each held-out client fold ONLY ===
 fold  n_test  base_rate  precision@20  precision@50  precision@100
    0   25520     0.2153          0.10          0.20           0.18
    1    4276     0.1284          0.05          0.08           0.05
    2    6675     0.6649          0.90          0.88           0.86
    3   16084     0.2743          0.65          0.38           0.41
    4   28966     0.1859          0.45          0.34           0.32

Mean fold P@20: 0.4300   (tie-break: score desc, seeded content-hash asc)
Mean fold P@50: 0.3760   (tie-break: score desc, seeded content-hash asc)
Mean fold P@100: 0.3640   (tie-break: score desc, seeded content-hash asc)

Frozen baseline receipt: e:\FlyRank-A

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The table below shows the top 20 rows the rule puts in front of the editor, each with a one-line "what would make it wrong" note computed from that row's own numbers.

In [3]:
# Section 3 — the top-20 review with a skeptic's eye

top = queue.head(20).copy()

def why_wrong(row):
    notes = []
    if row['pos_prev30'] > 20:
        notes.append(f"it sits deep (pos {row['pos_prev30']:.0f}) — a one-query shake-out could trip the decline label")
    if row['days_with_imp_prev30'] < 15:
        notes.append(f"traffic is intermittent ({row['days_with_imp_prev30']:.0f}/30 days) — one slow March week looks like decline")
    if row['clk_prev30'] == 0:
        notes.append(f"zero clicks on {row['imp_prev30']:.0f} impressions — the snippet it ranks on could change and the impressions vanish")
    elif row['clk_prev30'] < row['imp_prev30'] * 0.001:
        notes.append(f"fewer than 0.1% of impressions become clicks ({row['clk_prev30']:.0f}/{row['imp_prev30']:.0f}) — a CTR problem, so impressions are fragile")
    if row['imp_prev30'] >= 5000:
        notes.append("it is a big page — March could be a normal seasonal trough, not decay")
    if not notes:
        notes.append("the drop is real only if it persists past one month — one window is thin evidence")
    return 'Wrong if: ' + '; or '.join(notes) + '.'

def confidence(row):
    c = int(row['days_with_imp_prev30'] >= 25) + int(row['pos_prev30'] < 15) \
        + int(row['clk_prev30'] >= 10) + int(row['imp_prev30'] >= 1000)
    if c >= 3:
        return 'high'
    if c == 2:
        return 'medium'
    return 'low'

top['what_would_make_it_wrong'] = top.apply(why_wrong, axis=1)
top['confidence'] = top.apply(confidence, axis=1)
top['declined_in_march'] = top['is_declining'].map({1: 'YES', 0: 'no'})

print(f'Top-20 review — {top["is_declining"].sum()} of 20 actually declined in March ({top["is_declining"].mean():.0%}).')
print()
for i, r in top.reset_index(drop=True).iterrows():
    print(f'{i+1:>2}. action={r["action_label"]:15s} reason={r["reason_code"]:17s} '
          f'confidence={r["confidence"]:6s} imp30={r["imp_prev30"]:>7.0f} clk30={r["clk_prev30"]:>5.0f} '
          f'pos={r["pos_prev30"]:>6.1f} days={r["days_with_imp_prev30"]:>2.0f} '
          f'age={r["content_age_days"]:>4.0f}d declined={r["declined_in_march"]}')
    print(f'      {r["what_would_make_it_wrong"]}')

Top-20 review — 9 of 20 actually declined in March (45%).

 1. action=review_refresh  reason=stale_but_visible confidence=high   imp30= 204176 clk30=    2 pos=   4.8 days=30 age= 380d declined=YES
      Wrong if: fewer than 0.1% of impressions become clicks (2/204176) — a CTR problem, so impressions are fragile; or it is a big page — March could be a normal seasonal trough, not decay.
 2. action=review_refresh  reason=stale_but_visible confidence=high   imp30= 198339 clk30=    0 pos=   3.6 days=30 age= 380d declined=YES
      Wrong if: zero clicks on 198339 impressions — the snippet it ranks on could change and the impressions vanish; or it is a big page — March could be a normal seasonal trough, not decay.
 3. action=review_refresh  reason=stale_but_visible confidence=high   imp30= 195655 clk30=    1 pos=   3.5 days=30 age= 213d declined=YES
      Wrong if: fewer than 0.1% of impressions become clicks (1/195655) — a CTR problem, so impressions are fragile; or it is a big page — March 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# Section 4 — weak picks and the leakage hunt

print('=== Weak picks ===')
wrong = queue.head(20)[queue.head(20)['is_declining'] == 0]
print(f'Top-20 picks that did NOT decline in March: {len(wrong)}. They are the cost of ranking by size,')
print('not by decline probability — expected from a transparent baseline, and the model must beat it.')
print()

print('=== Leakage check ===')
rule_features = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
print('Rule features:', rule_features)
print('imp_last30 (the label window itself) in the rule?', 'imp_last30' in rule_features)
print('is_declining (the label) in the rule?', 'is_declining' in rule_features)
print()

# The future-window trap with the warehouse snapshot:
# dim_content.content_updated_date is the LAST update recorded at snapshot time (through July 2026).
upd = con.sql(f"SELECT SUM(CASE WHEN content_updated_date > DATE '2026-03-01' THEN 1 ELSE 0 END) AS fut,"
              f" COUNT(*) AS n FROM {DIM_CONTENT}").df().iloc[0]
print('=== Future-window trap in dim_content ===')
print(f"content_updated_date after the decision date (2026-03-01): {upd['fut']:,} of {upd['n']:,} rows "
      f"({upd['fut']/upd['n']*100:.1f}%)")
print('Using content_updated_date raw would leak the future for those pages — so the rule uses')
print('content_age_days (from content_created_date), which is fully knowable at the decision moment.')
print()

print('=== CSV hygiene ===')
csv_cols = pd.read_csv(os.path.join(OUT_DIR, 'baseline_action_score.csv'), nrows=1).columns.tolist()
print('CSV columns:', csv_cols)
print('Label or future-window column in the CSV?', any(c in csv_cols for c in ['imp_last30', 'is_declining']))

=== Weak picks ===
Top-20 picks that did NOT decline in March: 11. They are the cost of ranking by size,
not by decline probability — expected from a transparent baseline, and the model must beat it.

=== Leakage check ===
Rule features: ['imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
imp_last30 (the label window itself) in the rule? False
is_declining (the label) in the rule? False

=== Future-window trap in dim_content ===
content_updated_date after the decision date (2026-03-01): 383,714.0 of 519,606.0 rows (73.8%)
Using content_updated_date raw would leak the future for those pages — so the rule uses
content_age_days (from content_created_date), which is fully knowable at the decision moment.

=== CSV hygiene ===
CSV columns: ['content_hash_id', 'client_hash_id', 'fold', 'score', 'reason_code', 'action_label', 'imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
Label or future-window column in the CSV? False


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal checks with bucket tables and n, both with one-word verdicts (staleness CONFIRMED; volume CONFIRMED as opportunity-size)
- [x] One rule with a score, ONE reason code per row, and an action label
- [x] Rule scored against the `is_declining` label ONLY on held-out client folds (5-way hash split, every client on one side)
- [x] Frozen baseline: per-fold precision@20/50/100 + base rate + tie policy written to `work/outputs/baseline_folds_receipt.json`
- [x] Ranked queue written to work/outputs/baseline_action_score.csv from the notebook
- [x] Top-20 reviewed with "what would make it wrong" per row
- [x] No future-window or label-derived inputs (content_updated_date future-trap shown and avoided)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
